In [1]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

In [2]:
model = load_model(
    "../saved_models/best_finetuned_model.keras"
)

2026-05-29 22:44:42.391179: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-05-29 22:44:42.391410: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-29 22:44:42.391416: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-05-29 22:44:42.391708: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-29 22:44:42.391737: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
print(model.input_shape)

(None, 224, 224, 3)


In [4]:
for layer in model.layers:
    print(layer.name)

efficientnetb0
global_average_pooling2d
dropout
dense


In [5]:
img_path = "../test_images/diseased_leaf.jpg"

img = image.load_img(
    img_path,
    target_size=(224, 224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)

In [6]:
from tensorflow.keras.applications.efficientnet import preprocess_input

img_array = preprocess_input(img_array)

In [7]:
base_model = model.get_layer("efficientnetb0")

grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[
        base_model.get_layer("top_conv").output,
        model.output
    ]
)

AttributeError: The layer sequential has never been called and thus has no defined output.

In [8]:
dummy_input = tf.random.normal((1, 224, 224, 3))

_ = model(dummy_input)


In [9]:
print(model.inputs)
print(model.outputs)

[<KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=input_layer_1>]
[<KerasTensor shape=(None, 38), dtype=float32, sparse=False, ragged=False, name=keras_tensor_508>]


In [10]:
base_model = model.get_layer("efficientnetb0")

grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[
        base_model.get_layer("top_conv").output,
        model.outputs[0]
    ]
)

In [11]:
dummy_input = tf.random.normal((1, 224, 224, 3))
_ = model(dummy_input)

print(model.outputs)

[<KerasTensor shape=(None, 38), dtype=float32, sparse=False, ragged=False, name=keras_tensor_508>]


In [12]:
base_model = model.get_layer("efficientnetb0")

grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[
        base_model.get_layer("top_conv").output,
        model.outputs[0]
    ]
)

In [13]:
with tf.GradientTape() as tape:

    conv_outputs, predictions = grad_model(img_array)

    predicted_class = tf.argmax(predictions[0])

    loss = predictions[:, predicted_class]

/Users/vanshitasingh/Desktop/LeafDiseaseProject/venv/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


KeyError: "Exception encountered when calling Functional.call().\n\n\x1b[1m13352567760\x1b[0m\n\nArguments received by Functional.call():\n  • inputs=array([[[[179., 169., 168.],\n         [181., 171., 170.],\n         [184., 174., 173.],\n         ...,\n         [162., 146., 146.],\n         [115.,  99.,  99.],\n         [139., 123., 123.]],\n\n        [[177., 167., 166.],\n         [178., 168., 167.],\n         [178., 168., 167.],\n         ...,\n         [142., 126., 126.],\n         [175., 159., 159.],\n         [125., 109., 109.]],\n\n        [[178., 168., 167.],\n         [176., 166., 165.],\n         [174., 164., 163.],\n         ...,\n         [109.,  93.,  93.],\n         [157., 141., 141.],\n         [101.,  85.,  85.]],\n\n        ...,\n\n        [[183., 169., 169.],\n         [188., 174., 174.],\n         [193., 179., 179.],\n         ...,\n         [150., 135., 132.],\n         [154., 139., 136.],\n         [133., 118., 115.]],\n\n        [[189., 175., 175.],\n         [19

In [14]:
grads = tape.gradient(loss, conv_outputs)

pooled_grads = tf.reduce_mean(
    grads,
    axis=(0, 1, 2)
)

NameError: name 'loss' is not defined

In [15]:
with tf.GradientTape() as tape:

    conv_outputs, predictions = grad_model(img_array)

    predicted_class = tf.argmax(predictions[0])

    loss = predictions[:, predicted_class]

print(predictions.shape)
print(predicted_class.numpy())

KeyError: "Exception encountered when calling Functional.call().\n\n\x1b[1m13352567760\x1b[0m\n\nArguments received by Functional.call():\n  • inputs=array([[[[179., 169., 168.],\n         [181., 171., 170.],\n         [184., 174., 173.],\n         ...,\n         [162., 146., 146.],\n         [115.,  99.,  99.],\n         [139., 123., 123.]],\n\n        [[177., 167., 166.],\n         [178., 168., 167.],\n         [178., 168., 167.],\n         ...,\n         [142., 126., 126.],\n         [175., 159., 159.],\n         [125., 109., 109.]],\n\n        [[178., 168., 167.],\n         [176., 166., 165.],\n         [174., 164., 163.],\n         ...,\n         [109.,  93.,  93.],\n         [157., 141., 141.],\n         [101.,  85.,  85.]],\n\n        ...,\n\n        [[183., 169., 169.],\n         [188., 174., 174.],\n         [193., 179., 179.],\n         ...,\n         [150., 135., 132.],\n         [154., 139., 136.],\n         [133., 118., 115.]],\n\n        [[189., 175., 175.],\n         [19

In [16]:
print(type(model))
print(type(model.layers[0]))

<class 'keras.src.models.sequential.Sequential'>
<class 'keras.src.models.functional.Functional'>


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,187,927 (27.42 MB)

 Trainable params: 1,544,838 (5.89 MB)

 Non-trainable params: 2,553,411 (9.74 MB)

 Optimizer params: 3,089,678 (11.79 MB)

In [19]:
base_model = model.layers[0]

print(base_model.name)

for layer in base_model.layers[-10:]:
    print(layer.name)

efficientnetb0
block7a_se_squeeze
block7a_se_reshape
block7a_se_reduce
block7a_se_expand
block7a_se_excite
block7a_project_conv
block7a_project_bn
top_conv
top_bn
top_activation
